# v5 Local Experiment — MiniCPM-V 4.6 QLoRA

이 노트북은 `baseline_v4_recommended.ipynb`의 **group split / 객관식 prompt / 1 epoch / LoRA r=8 / lr=5e-5**를 최대한 유지하면서 백본만 MiniCPM-V 4.6으로 바꿔 실험합니다.

실험 스위치:

- `A_backbone_16x`: language-only QLoRA, 16x visual compression — 가장 빠른 1차 실험
- `B_resolution_4x`: A와 동일하지만 4x — 작은 글자/OCR 정보량 가설
- `C_projector_4x`: 4x + multimodal projector를 열어 adaptation — image→text alignment 가설

모든 실험은 validation의 `a/b/c/d` score를 CSV로 저장합니다. `output/v4_valid_scores.csv`가 있으면 v4와 오답 overlap 및 soft-voting ensemble도 바로 계산합니다.

> MiniCPM-V 4.6은 Transformers 5.7+ 계열이므로 v4 환경과 **별도 conda/venv**를 권장합니다.


In [ ]:
from pathlib import Path
import os, json, re, random, shutil, subprocess, sys, math
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "dataset"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# 오프라인이면 다운로드해 둔 모델 경로를 사용하세요.
LOCAL_MODEL_DIR = ROOT / "downloads" / "models" / "MiniCPM-V-4.6"
OFFLINE = True
MODEL_ID = str(LOCAL_MODEL_DIR) if OFFLINE else "openbmb/MiniCPM-V-4.6"

SEED = 42
VAL_FRAC = 0.10
TRAIN_LIMIT = None       # smoke test: 200~500 / 본 실험: None

# 실험 하나만 바꿔서 재실행합니다.
EXPERIMENT = "A_backbone_16x"
EXPERIMENTS = {
    "A_backbone_16x": dict(
        downsample_mode="16x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=True,
    ),
    "B_resolution_4x": dict(
        downsample_mode="4x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=True,
    ),
    "C_projector_4x": dict(
        downsample_mode="4x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=False,
    ),
}
CFG = EXPERIMENTS[EXPERIMENT]
RUN_DIR = OUTPUT_DIR / f"v5_minicpm46_{EXPERIMENT}"
ADAPTER_DIR = RUN_DIR / "adapter"
MERGED_DIR = RUN_DIR / "merged"
LF_DATA_DIR = RUN_DIR / "lf_data"
for p in [RUN_DIR, LF_DATA_DIR]: p.mkdir(parents=True, exist_ok=True)

if OFFLINE:
    assert LOCAL_MODEL_DIR.exists(), f"모델이 없습니다: {LOCAL_MODEL_DIR}"

print("experiment:", EXPERIMENT)
print("model     :", MODEL_ID)
print("run dir   :", RUN_DIR)
print("settings  :", CFG)


## 0. 환경

권장 새 환경(최초 1회):

```bash
conda create -n minicpm46 python=3.11 -y
conda activate minicpm46

# 현재 CUDA에 맞는 PyTorch를 설치하세요. 공식 MiniCPM-V 4.6 fine-tuning 예시는 torch 2.8 / torchvision 0.23을 사용합니다.
pip install torch==2.8.0 torchvision==0.23.0
pip install transformers==5.7.0 accelerate==1.13.0 peft==0.18.1 trl==0.24.0 \
            bitsandbytes datasets pandas pillow av pyyaml

git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
cd LLaMA-Factory
pip install -e ".[torch,metrics,minicpm_v]"
```

오프라인 PC라면 이 환경과 `downloads/models/MiniCPM-V-4.6` 모델을 미리 준비한 뒤 아래 셀부터 실행하세요.


In [ ]:
import torch, transformers
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."

from packaging.version import Version
assert Version(transformers.__version__) >= Version("5.7.0"), "MiniCPM-V 4.6은 transformers>=5.7.0 권장"
assert shutil.which("llamafactory-cli"), "llamafactory-cli가 없습니다. 위 환경 설치를 먼저 하세요."


## 1. v4와 동일한 group split 생성


In [ ]:
random.seed(SEED)
np.random.seed(SEED)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

if TRAIN_LIMIT is not None:
    train_df = train_df.sample(n=min(TRAIN_LIMIT, len(train_df)), random_state=SEED).reset_index(drop=True)

def normalize_question(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def make_group_split(df, val_frac=0.10, seed=42, trials=100):
    groups = {}
    for idx, q in enumerate(df["question"].map(normalize_question)):
        groups.setdefault(q, []).append(idx)

    group_items = list(groups.items())
    target_n = int(round(len(df) * val_frac))
    overall = df["answer"].astype(str).str.lower().value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)

    best = None
    for t in range(trials):
        rng = random.Random(seed + t)
        items = group_items.copy()
        rng.shuffle(items)
        val_idx = []
        for _, idxs in items:
            if len(val_idx) >= target_n:
                break
            val_idx.extend(idxs)
        val_idx = sorted(set(val_idx))
        val_prop = df.iloc[val_idx]["answer"].astype(str).str.lower().value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)
        score = float((val_prop - overall).abs().sum()) + abs(len(val_idx) - target_n) / len(df)
        if best is None or score < best[0]:
            best = (score, val_idx)

    val_idx = set(best[1])
    train_idx = [i for i in range(len(df)) if i not in val_idx]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[sorted(val_idx)].reset_index(drop=True)

train_subset, valid_subset = make_group_split(train_df, VAL_FRAC, SEED)
print("train/valid:", len(train_subset), len(valid_subset))
print(valid_subset["answer"].astype(str).str.lower().value_counts(normalize=True).sort_index())

# 다른 모델/노트북과 validation 행을 맞추는 키
valid_keys = valid_subset[[c for c in ["id", "path", "question", "answer"] if c in valid_subset.columns]].copy()
valid_keys.to_csv(RUN_DIR / "valid_keys.csv", index=False)


## 2. LLaMA-Factory용 multimodal SFT 데이터 생성


In [ ]:
SYSTEM_INSTRUCT = (
    "이미지를 보고 객관식 질문에 답하세요. "
    "필요하면 이미지 속 글자, 숫자, 표지판, 가격, 상호명 등 세부 정보를 주의 깊게 읽으세요. "
    "최종 답은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

def build_mc_prompt(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n"
        "정답:"
    )

def to_lf_records(df, with_answer=True):
    records = []
    for _, row in df.iterrows():
        img_path = (DATA_DIR / str(row["path"])).resolve()
        assert img_path.exists(), img_path
        msgs = [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": "<image>\n" + build_mc_prompt(row)},
        ]
        if with_answer:
            msgs.append({"role": "assistant", "content": str(row["answer"]).strip().lower()})
        records.append({"messages": msgs, "images": [str(img_path)]})
    return records

train_json = LF_DATA_DIR / "kaggle_v5_train.json"
valid_json = LF_DATA_DIR / "kaggle_v5_valid.json"
train_json.write_text(json.dumps(to_lf_records(train_subset), ensure_ascii=False, indent=2), encoding="utf-8")
valid_json.write_text(json.dumps(to_lf_records(valid_subset), ensure_ascii=False, indent=2), encoding="utf-8")

info = {
    "kaggle_v5_train": {
        "file_name": train_json.name,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "role", "content_tag": "content",
            "user_tag": "user", "assistant_tag": "assistant", "system_tag": "system"
        },
    },
    "kaggle_v5_valid": {
        "file_name": valid_json.name,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "role", "content_tag": "content",
            "user_tag": "user", "assistant_tag": "assistant", "system_tag": "system"
        },
    },
}
(LF_DATA_DIR / "dataset_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", train_json, valid_json, LF_DATA_DIR / "dataset_info.json", sep="\n")


## 3. 4-bit QLoRA 설정 생성


In [ ]:
import yaml

train_cfg = {
    # model
    "model_name_or_path": MODEL_ID,
    "trust_remote_code": True,
    "flash_attn": "auto",

    # method
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_target": "all",
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "freeze_vision_tower": CFG["freeze_vision_tower"],
    "freeze_multi_modal_projector": CFG["freeze_multi_modal_projector"],

    # QLoRA
    "quantization_bit": 4,
    "quantization_method": "bnb",
    "double_quantization": True,

    # data
    "dataset_dir": str(LF_DATA_DIR.resolve()),
    "dataset": "kaggle_v5_train",
    "template": "minicpm_v_4_6",
    "enable_thinking": False,
    "cutoff_len": 4096,
    "packing": False,
    "overwrite_cache": True,
    "preprocessing_num_workers": 4,

    # train — v4와 최대한 동일하게
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-5,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "linear",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "bf16": bool(torch.cuda.is_bf16_supported()),
    "fp16": not bool(torch.cuda.is_bf16_supported()),

    # output
    "output_dir": str(ADAPTER_DIR.resolve()),
    "logging_steps": 5,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "plot_loss": True,
    "overwrite_output_dir": True,
    "report_to": "none",
}

TRAIN_YAML = RUN_DIR / "train.yaml"
TRAIN_YAML.write_text(yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(TRAIN_YAML.read_text(encoding="utf-8"))


## 4. 학습

MiniCPM-V 4.6은 `DOWNSAMPLE_MODE=16x/4x`를 사용합니다. `16x`는 빠르고, `4x`는 visual token이 더 많아 작은 글자/OCR에 유리한지 검증하는 실험입니다.


In [ ]:
RUN_TRAIN = True

env = os.environ.copy()
env["DOWNSAMPLE_MODE"] = CFG["downsample_mode"]
env["DISABLE_VERSION_CHECK"] = "1"
if OFFLINE:
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"

cmd = ["llamafactory-cli", "train", str(TRAIN_YAML)]
print(" ".join(cmd))
print("DOWNSAMPLE_MODE=", env["DOWNSAMPLE_MODE"])
if RUN_TRAIN:
    subprocess.run(cmd, env=env, check=True)


## 5. LoRA merge — 평가는 merged BF16 모델로 단순화


In [ ]:
merge_cfg = {
    "model_name_or_path": MODEL_ID,
    "adapter_name_or_path": str(ADAPTER_DIR.resolve()),
    "template": "minicpm_v_4_6",
    "finetuning_type": "lora",
    "trust_remote_code": True,
    "export_dir": str(MERGED_DIR.resolve()),
    "export_size": 2,
    "export_device": "auto",
    "export_legacy_format": False,
}
MERGE_YAML = RUN_DIR / "merge.yaml"
MERGE_YAML.write_text(yaml.safe_dump(merge_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(MERGE_YAML.read_text(encoding="utf-8"))

RUN_MERGE = True
if RUN_MERGE:
    # 중요: merge 시 quantization_bit를 넣지 않습니다.
    subprocess.run(["llamafactory-cli", "export", str(MERGE_YAML)], env=env, check=True)


## 6. Validation a/b/c/d score 저장

`generate()` 문자열 파싱 대신 assistant 첫 토큰의 `a/b/c/d` logits을 비교합니다. 시작 전에 몇 샘플을 generate해 **첫 출력이 실제로 a/b/c/d인지 sanity check**합니다.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from tqdm.auto import tqdm

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(
    str(MERGED_DIR),
    local_files_only=OFFLINE,
)
model = AutoModelForImageTextToText.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    local_files_only=OFFLINE,
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print("model device:", MODEL_DEVICE)

CHOICES = ["a", "b", "c", "d"]
choice_ids = []
for c in CHOICES:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    print(c, ids)
    if len(ids) != 1:
        raise RuntimeError(f"{c!r}가 단일 토큰이 아닙니다. 이 경우 sequence scoring으로 바꾸세요: {ids}")
    choice_ids.append(ids[0])
choice_ids = torch.tensor(choice_ids, device=MODEL_DEVICE)


In [ ]:
def build_messages(row):
    img_path = (DATA_DIR / str(row["path"])).resolve()
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "path": str(img_path)},
            {"type": "text", "text": build_mc_prompt(row)},
        ]},
    ]

@torch.inference_mode()
def smoke_generate(df, n=3):
    for _, row in df.head(n).iterrows():
        inputs = processor.apply_chat_template(
            build_messages(row), tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
            downsample_mode=CFG["downsample_mode"], max_slice_nums=36,
        ).to(MODEL_DEVICE)
        out = model.generate(
            **inputs,
            downsample_mode=CFG["downsample_mode"],
            max_new_tokens=8,
            do_sample=False,
        )
        new_ids = out[0, inputs["input_ids"].shape[1]:]
        text = processor.decode(new_ids, skip_special_tokens=True).strip()
        print("gold=", str(row["answer"]).lower(), "generated=", repr(text))

smoke_generate(valid_subset, 3)
print("첫 글자가 a/b/c/d가 아니면 아래 logit scoring을 그대로 쓰지 말고 prompt/template부터 확인하세요.")


In [ ]:
@torch.inference_mode()
def score_df(df):
    rows = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"score {EXPERIMENT}"):
        inputs = processor.apply_chat_template(
            build_messages(row), tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
            downsample_mode=CFG["downsample_mode"], max_slice_nums=36,
        ).to(MODEL_DEVICE)

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            out = model(**inputs, downsample_mode=CFG["downsample_mode"])

        last_idx = int(inputs["attention_mask"][0].sum().item()) - 1
        next_logits = out.logits[0, last_idx]
        s = next_logits.index_select(0, choice_ids).float()
        logp = torch.log_softmax(s, dim=0).cpu().numpy()
        pred_idx = int(np.argmax(logp))

        rec = {
            "row_idx": i,
            "path": str(row["path"]),
            "question": str(row["question"]),
            "gold": str(row["answer"]).strip().lower() if "answer" in row else None,
            "pred": CHOICES[pred_idx],
        }
        for c, v in zip(CHOICES, logp):
            rec[f"logp_{c}"] = float(v)
        rows.append(rec)
    return pd.DataFrame(rows)

valid_scores = score_df(valid_subset)
valid_acc = (valid_scores["pred"] == valid_scores["gold"]).mean()
VALID_SCORE_PATH = RUN_DIR / "valid_scores.csv"
valid_scores.to_csv(VALID_SCORE_PATH, index=False)
print(f"Validation accuracy: {valid_acc:.5f}")
print("saved:", VALID_SCORE_PATH)
valid_scores.head()


## 7. v4와 오답 overlap / ensemble


In [ ]:
# baseline_v4_with_scores.ipynb를 먼저 실행하면 기본 위치에 생성됩니다.
V4_SCORE_PATH = OUTPUT_DIR / "v4_valid_scores.csv"

if V4_SCORE_PATH.exists():
    v4 = pd.read_csv(V4_SCORE_PATH)
    v5 = valid_scores.copy()
    key = ["path", "question"]
    m = v4.merge(v5, on=key, suffixes=("_v4", "_v5"), validate="one_to_one")

    v4_acc = (m.pred_v4 == m.gold_v4).mean()
    v5_acc = (m.pred_v5 == m.gold_v5).mean()
    both_wrong = ((m.pred_v4 != m.gold_v4) & (m.pred_v5 != m.gold_v5)).mean()
    v4_only_wrong = ((m.pred_v4 != m.gold_v4) & (m.pred_v5 == m.gold_v5)).mean()
    v5_only_wrong = ((m.pred_v4 == m.gold_v4) & (m.pred_v5 != m.gold_v5)).mean()

    # log-prob soft voting. alpha=0.5부터 보고 필요하면 validation에서만 탐색합니다.
    alphas = np.linspace(0, 1, 21)
    ens = []
    for alpha in alphas:
        mat = np.stack([
            alpha*m[f"logp_{c}_v4"].to_numpy() + (1-alpha)*m[f"logp_{c}_v5"].to_numpy()
            for c in CHOICES
        ], axis=1)
        pred = np.array(CHOICES)[mat.argmax(1)]
        acc = (pred == m.gold_v4.to_numpy()).mean()
        ens.append((alpha, acc))

    print(f"v4 acc       : {v4_acc:.5f}")
    print(f"v5 acc       : {v5_acc:.5f}")
    print(f"both wrong   : {both_wrong:.5f}")
    print(f"v4-only wrong: {v4_only_wrong:.5f}  <- v5가 구한 문제")
    print(f"v5-only wrong: {v5_only_wrong:.5f}  <- v4가 구한 문제")
    print("best ensemble(alpha=v4 weight):", max(ens, key=lambda x: x[1]))
else:
    print("v4 score 파일 없음:", V4_SCORE_PATH)
    print("같이 제공한 baseline_v4_with_scores.ipynb를 실행한 뒤 다시 이 셀을 실행하세요.")


## 8. Test score + submission


In [ ]:
# test에는 answer가 없으므로 score_df를 약간 감싸서 사용합니다.
def score_test(df):
    temp = df.copy()
    if "answer" not in temp.columns:
        temp["answer"] = ""
    return score_df(temp)

RUN_TEST = True
if RUN_TEST:
    test_scores = score_test(test_df)
    test_scores.to_csv(RUN_DIR / "test_scores.csv", index=False)
    preds = test_scores["pred"].tolist()

    sample_path = DATA_DIR / "sample_submission.csv"
    if sample_path.exists():
        submission = pd.read_csv(sample_path)
        submission["answer"] = preds
    else:
        submission = pd.DataFrame({"id": test_df["id"], "answer": preds})

    sub_path = RUN_DIR / "submission.csv"
    submission.to_csv(sub_path, index=False)
    print("saved:", sub_path)
    display(submission.head())


## 실험 실행 순서

1. `A_backbone_16x` — 가장 먼저. v4보다 얼마나 다르게 틀리는지 확인
2. `B_resolution_4x` — A에서 OCR/작은 글씨 오답이 많으면 실행
3. `C_projector_4x` — B에서도 visual alignment 문제가 남을 때 실행

판단 기준은 단일 accuracy만이 아닙니다.

- v5 accuracy
- `v4-only wrong`: v4는 틀렸는데 v5가 맞힌 비율
- `v5-only wrong`: v5는 틀렸는데 v4가 맞힌 비율
- validation soft-voting ensemble의 상승 여부

`C_projector_4x`는 A/B보다 실험 변수가 하나 더 많으므로 A/B 결과가 나온 뒤 진행하는 것을 권장합니다.
